# H-R Diagram of a Globular and Open Clusters

## Introduction

The Hertzsprung-Russell (H-R) Diagram is used to classify stars according to their luminosities and colors. The majority of stars, including our Sun, are found in such a diagram along a region called the main sequence. Main sequence stars vary widely in effective temperatures, but the hotter and larger they are, the more luminous they are, hence the main sequence follows a band running from the bottom right to the top left of the diagram. These stars are fusing hydrogen into helium in their cores, and they spend the bulk of their existence in this phase. Other major groups found in the H-R diagram are the giants and supergiants (luminous stars that have evolved off the main sequence) and the white dwarfs. The positions of stars in the H-R diagram allow us to infer their individual properties as well as characteristics of the cluster they belong to (age, metallicity, etc.)

In this tutorial, you will build the H-R diagrams of a star clusters and compare them.

The notebook follows these steps:
- **Calibrate** the images in the three filters.
- **Perform aperture photometry** on the stars in the stacks.
- **Plot** the H-R diagram of the clusters.

## Load images

In [ ]:
import helper

# Find images from telescope
target = "M67"
image_paths = helper.find_files(target)

## Calibrate images

In [ ]:
from astropy.io import fits
from eloy import calibration
import tqdm

filters = ["RP", "G", "BP"]
stacks = {}

# Master calibration frames
print("Creating master bias and dark frames...\n")
bias = calibration.master_bias(image_paths["bias"])
dark = calibration.master_dark(bias, image_paths["dark"])

for filter in filters:
    print(f"Processing {filter} filter...")
    print(f"(1/3) Creating master flat frame for {filter} filter...")
    flat = calibration.master_flat(bias, dark, image_paths["flat"][filter])

    print(f"(2/3) Calibrating {filter} frames using master frames...")
    calibrated_images = []
    # Calibrate light frames
    for file in tqdm.tqdm(image_paths["light"][filter]):
        # Get the raw data and header information
        data = fits.getdata(file)
        header = fits.getheader(file)
        exposure = header["EXPTIME"]
        collecting_area = header["APTAREA"]

        # Calibrate
        calibrated_data = calibration.calibrate(data, exposure, dark, flat, bias)
        calibrated_images.append(calibrated_data / exposure / collecting_area)

    print(f"(3/3) Stacking {filter} frames...", end=" ")
    images_median = calibration.easy_median(calibrated_images)
    stacks[filter] = images_median
    print(f"Finished {filter} images.\n")

print("All done!")

## Plot images

In [ ]:
import matplotlib.pyplot as plt
from eloy import viz

# Visualize a raw image
raw_image = fits.getdata(image_paths["light"]["G"][0])
plt.figure(figsize=(15, 15))
plt.imshow(viz.z_scale(raw_image, c=0.25), origin="lower", cmap="gray")
plt.title("Raw image (G filter)")

# Visualize the master calibration frames
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for i, image in enumerate((bias, dark, flat)):
    axs[i].imshow(viz.z_scale(image, c=0.25), origin="lower", cmap="gray")
    axs[i].set_title("Master " + ["bias", "dark", "flat"][i])

# Visualize the stacked images for each filter
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for i, (filter, stack) in enumerate(stacks.items()):
    axs[i].imshow(viz.z_scale(stack, c=0.25), origin="lower", cmap="gray")
    axs[i].set_title("Stacked " + filter + " image")

In [ ]:
# Save stacks for later use
for filter, stack in stacks.items():
    fits.writeto(f"{target}_{filter}_stack.fits", stack, overwrite=True)
    print(f"Saved {filter} stack as {target}_{filter}_stack.fits")

## Make color image

In [ ]:
from astropy.visualization import ManualInterval, make_rgb, LogStretch
import numpy as np


r = stacks["RP"] - np.median(stacks["RP"])
g = stacks["G"] - np.median(stacks["G"])
b = stacks["BP"] - np.median(stacks["BP"])

maximum = 0.0
for img in [r, g, b]:
    val = np.percentile(img, 99.9)
    if val > maximum:
        maximum = val
rgb = make_rgb(
    image_r=r,
    image_g=g,
    image_b=b,
    interval=ManualInterval(vmin=0, vmax=maximum),
    stretch=LogStretch(a=1000),
    output_dtype=float,
)

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_xlabel("Pixels")
ax.set_ylabel("Pixels")
_ = ax.imshow(rgb, origin="lower")

The three images cover the same field of view but appear slightly different. The **G** stack looks brightest because its broadband filter captures photons across the widest wavelength range. In **BP**, hot blue stars appear relatively bright compared to cool red ones, while the opposite is true in **RP**. This differential brightness between filters is what encodes the color of each star, and it is precisely the information we will exploit to build the H-R diagram.

## Aperture photometry

The last step is to measure the brightness of each star through **aperture photometry**: we sum the pixel values inside a circular aperture centred on each star. The `eloy` package provides an `aperture_photometry` function that handles this automatically, we just supply the star positions and the aperture radius.

In [ ]:
from astropy.io import fits

# Load stacks from disk
stack_rp = fits.getdata(f"{target}_RP_stack.fits")
stack_g = fits.getdata(f"{target}_G_stack.fits")
stack_bp = fits.getdata(f"{target}_BP_stack.fits")

stacks = {"RP": stack_rp, "G": stack_g, "BP": stack_bp}

print("Loaded stacks from disk:")
for filter, stack in stacks.items():
    print(f"  {filter}: {stack.shape[0]} x {stack.shape[1]} pixels")

In [ ]:
# Create a combined image for star detection
stacks_sum = stacks["RP"] + stacks["G"] + stacks["BP"]

# Detect stars in the image
coords = helper.find_stars(stacks_sum)
print(f"Detected {len(coords)} stars")

# Measure FWHM of the detected stars
fwhm = helper.measure_fwhm(stacks["G"], coords)
print(f"FWHM: {fwhm:.2f} pixels (~average width of a star in the image)")

# Plot image withs detected stars
fig, ax = plt.subplots(figsize=(12, 12))
ax.set_xlabel("Pixels")
ax.set_ylabel("Pixels")
_ = ax.imshow(rgb, origin="lower")
ax.plot(coords[:, 0], coords[:, 1], "x", color="red", markersize=10)


In [ ]:
from eloy import utils

CUTOUT = 120

# Visualize cutouts of a detected star with annuli for photometry
# The inner annulus (orange) is used for aperture photometry, while the outer annulus (blue) is used to estimate the background flux.
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for i, (filter, stack) in enumerate(stacks.items()):
    target_cutout = utils.cutout(stacks[filter], [coords[len(coords) // 2]], CUTOUT)[0]
    axs[i].imshow(viz.z_scale(target_cutout), cmap="Greys_r", origin="lower")
    axs[i].set_title(filter)

    for r in np.array((3, 10)) * fwhm:
        annulus_aperture = plt.Circle(
            (CUTOUT / 2, CUTOUT / 2),
            r,
            color=f"{'orange' if r == 3 * fwhm else 'steelblue'}",
            fill=True,
            label="Inner aperture" if r == 3 * fwhm else "Outer aperture",
            alpha=0.25,
        )
        axs[i].add_artist(annulus_aperture)
    axs[i].legend(loc="upper right")

In [ ]:
from eloy import photometry

# Perform aperture photometry on the detected stars in each filter, using the inner annulus for the aperture and the outer annulus for background estimation.
phots = {}
for filter, stack in stacks.items():
    print(f"Processing filter {filter}...")
    data = stacks[filter]
    bkg = photometry.annulus_sigma_clip_median(
        data, coords, r_in=fwhm * 5, r_out=fwhm * 10
    )
    # print(f"Background median for filter {filter}: {np.median(bkg):.2f}\n")
    phots[filter] = (
        photometry.aperture_photometry(data, coords, radii=[fwhm * 3])
        - bkg[:, None] * np.pi * (fwhm * 3) ** 2
    )

print("Photometry complete for all filters!")

## Plotting the H-R diagram

With fluxes measured in all three filters, we convert them to apparent magnitudes using $m = -2.5 \log_{10}(F) - zero\,point$ and plot color (BP − RP) against absolutemagnitude (G). This is the H-R diagram of the cluster.

In [ ]:
# calculate apparent magnitudes
zero_points = {"RP": -26.136, "G": -26.592, "BP": -26.139}
mags = {f: -2.5 * np.log10(phots[f]) - zero_points[f] for f in stacks.keys()}

m_g = mags["G"][:, 0]
m_bp = mags["BP"][:, 0]
m_rp = mags["RP"][:, 0]

# find distance to cluster (from literature) and calculate absolute magnitudes
distance = # parsecs, LOOKUP THIS VALUE!
dm = 5.0 * np.log10(distance) - 5.0  # distance modulus

# calculate absolute magnitudes and color
M_G = m_g - dm
color = m_bp - m_rp

# plotting
plt.figure(figsize=(8, 10))
plt.plot(color, M_G, ".", color="gray", alpha=0.5)
plt.xlabel("BP - RP")
plt.ylabel("Absolute G magnitude")
plt.title(f"H-R Diagram of {target}")
plt.minorticks_on()
plt.gca().invert_yaxis()

The y-axis shows the G magnitude (brighter stars at the top, since the magnitude scale runs from bright to faint). The x-axis shows the color index BP - RP: a small or negative value indicates a hot blue star, while a large positive value indicates a cool red one.

In a well-populated diagram you should be able to identify:

- A **main sequence** running diagonally from the upper left (bright, hot) to the lower right (faint, cool).
- A **turn-off point** where the most massive remaining main-sequence stars begin to leave the sequence. Its position encodes the age of the cluster: the brighter (more massive) the turn-off, the younger the cluster.
- A **red giant branch** extending upward and to the right of the turn-off, populated by stars that have exhausted the hydrogen in their cores and are now burning hydrogen in a shell around the core.

When you compare the globular cluster and the open cluster, you will notice that the turn-off point sits at very different luminosities, reflecting the much larger age difference between the two types of clusters.

## The exercise

Repeat the steps above for M45, and compare the resulting H-R diagrams. What do you notice about the turn-off points of the two clusters? What does this tell you about their ages?
